<a href="https://colab.research.google.com/github/Kevin-1chuli/Transformer-Overload-Prediction-System/blob/main/Transformer_Overload.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("Google drive mounted👍")

Mounted at /content/drive
Google drive mounted👍


In [2]:
!pip install -q xgboost scikit-learn

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple
import warnings
import pickle
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("✓ Libraries imported!")

✓ Libraries imported!


In [4]:
class Config:
    """All settings in one place - UPDATE PATHS HERE!"""


    BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/datasets"
    METADATA_PATH = os.path.join(BASE_PATH, "metadata.csv")
    IMP_FOLDER = os.path.join(BASE_PATH, "imp_csv")
    OUTPUT_DIR = "/content/transformer_outputs"


    ZIP_CODE_COL = 'zip_code'
    POWER_COLS = ['p1', 'p2', 'p3', 'p4', 'p5', 'p6']

    # Modeling parameters
    PREDICTION_HORIZON = 1  # hours ahead
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15

    # Overload detection
    OVERLOAD_METHOD = 'statistical'  # 'statistical' or 'capacity'
    STATISTICAL_PERCENTILE = 95
    USE_SEASONAL_THRESHOLDS = True

    # Feature engineering
    LAG_HOURS = [1, 2, 3, 6, 24, 48, 168]
    ROLLING_WINDOWS = [3, 6, 12, 24, 168]

    # Model
    MODEL_TYPE = 'xgboost'

    # Testing (set to 3 for quick test, None for all data)
    SAMPLE_TRANSFORMERS = None  # or 3 for testing

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

print("✓ Configuration loaded!")
print(f"  Metadata: {Config.METADATA_PATH}")
print(f"  Data: {Config.IMP_FOLDER}")
print(f"  Output: {Config.OUTPUT_DIR}")

✓ Configuration loaded!
  Metadata: /content/drive/MyDrive/Colab Notebooks/datasets/metadata.csv
  Data: /content/drive/MyDrive/Colab Notebooks/datasets/imp_csv
  Output: /content/transformer_outputs


In [5]:
def load_data(folder_path, file_name):
    """Load CSV or Excel file"""
    file_path = os.path.join(folder_path, file_name)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_name} not found at {file_path}")

    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.csv':
        return pd.read_csv(file_path)
    elif ext in ['.xls', '.xlsx']:
        return pd.read_excel(file_path)
    else:
        raise ValueError(f"Unsupported file type: {ext}")


def extract_user_id_from_filename(filename):
    """Extract user ID from filename"""
    base_name = os.path.splitext(filename)[0]

    if 'imp_' in base_name:
        user_id = base_name.replace('imp_', '').replace('_imp', '')
    else:
        user_id = base_name

    return user_id


print("✓ Loading functions ready!")

✓ Loading functions ready!


In [6]:
print("\n" + "="*80)
print("STEP 1: LOADING METADATA")
print("="*80)

# Loading metadata
meta_data = load_data(Config.BASE_PATH, "metadata.csv")
print(f"✓ Loaded metadata: {len(meta_data)} rows")
print(f"\nColumns: {meta_data.columns.tolist()}")
print(f"\nFirst rows:")
print(meta_data.head())

# Cleaning metadata
meta_data = meta_data.dropna(subset=[Config.ZIP_CODE_COL])
print(f"\n✓ After removing null zip_codes: {len(meta_data)} rows")

# Converting dates
date_cols = ['start_date', 'end_date', 'contract_start_date', 'contract_end_date']
for col in date_cols:
    if col in meta_data.columns:
        meta_data[col] = pd.to_datetime(meta_data[col], errors='coerce')

# Calculating max power per user
meta_data['max_power'] = meta_data[Config.POWER_COLS].max(axis=1)
print(f"\n✓ Max power statistics:")
print(meta_data['max_power'].describe())

# Group by zip_code (transformer)
zip_group = meta_data.groupby(Config.ZIP_CODE_COL).agg({
    'max_power': ['sum', 'mean', 'count'],
}).reset_index()

zip_group.columns = [
    Config.ZIP_CODE_COL,
    'total_max_power',
    'avg_max_power',
    'num_users'
]

# Calculate transformer capacity (80% of total max power)
zip_group['transformer_capacity_kW'] = zip_group['total_max_power'] * 0.8

print(f"\n✓ Transformer summary:")
print(zip_group.head(10))
print(f"\nTotal transformers: {len(zip_group)}")


STEP 1: LOADING METADATA
✓ Loaded metadata: 25559 rows

Columns: ['user', 'start_date', 'end_date', 'length_days', 'length_years', 'potential_samples', 'actual_samples', 'missing_samples_abs', 'missing_samples_pct', 'contract_start_date', 'contract_end_date', 'contracted_tariff', 'self_consumption_type', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'province', 'municipality', 'zip_code', 'cnae']

First rows:
                                                user            start_date  \
0  00000c5a448d9faa097b761cc98036d45a4e7d36032903...  2022-05-30T01:00:00Z   
1  0001b3b2f18c01c62ed9b2a87de7b4e33e7836f786f790...  2017-05-31T01:00:00Z   
2  0003de2700e20a1681d69fe287441d9041407a7698d5c8...  2017-05-31T01:00:00Z   
3  0004150214d14a2b2e6f7075531e661cf465b27ec4d0d5...  2018-07-12T01:00:00Z   
4  000721f0fc6ccf02ae24b67393979513171f2abc119af0...  2017-05-30T01:00:00Z   

               end_date  length_days  length_years  potential_samples  \
0  2022-06-05T00:00:00Z          6.0      0.016427    

In [7]:
print("\n" + "="*80)
print("STEP 2: LOADING USER FILES")
print("="*80)

# Get all user files
all_files = [f for f in os.listdir(Config.IMP_FOLDER) if f.endswith('.csv')]
print(f"✓ Found {len(all_files)} user files")

# Create mapping
user_file_map = {}
for filename in all_files:
    user_id = extract_user_id_from_filename(filename)
    user_file_map[user_id] = filename

print(f"✓ Created user mapping for {len(user_file_map)} files")

# Inspect sample file
sample_file = all_files[0]
sample_path = os.path.join(Config.IMP_FOLDER, sample_file)
sample_data = pd.read_csv(sample_path)

print(f"\n✓ Sample file: {sample_file}")
print(sample_data.head())
print(f"\nColumns: {sample_data.columns.tolist()}")
print(f"Shape: {sample_data.shape}")


STEP 2: LOADING USER FILES
✓ Found 3881 user files
✓ Created user mapping for 3881 files

✓ Sample file: 3ba39a93e651985be4ae9fe7dbccdc7e99052fd9c7bc4d4e6cafd1792b8e23e3.csv
             timestamp    kWh  imputed
0  2017-05-31 01:00:00  2.490        0
1  2017-05-31 02:00:00  2.311        0
2  2017-05-31 03:00:00  2.012        0
3  2017-05-31 04:00:00  0.150        0
4  2017-05-31 05:00:00  0.201        0

Columns: ['timestamp', 'kWh', 'imputed']
Shape: (24120, 3)


In [8]:
def calculate_imputation_features(df_transformer):
    """Calculate imputation-related features"""
    imputed_cols = [col for col in df_transformer.columns if 'imputed' in col]

    if not imputed_cols:
        df_transformer['imputation_rate'] = 0
        df_transformer['num_imputed_households'] = 0
        df_transformer['total_households'] = 0
        return df_transformer

    df_transformer['num_imputed_households'] = df_transformer[imputed_cols].sum(axis=1)
    df_transformer['total_households'] = len(imputed_cols)
    df_transformer['imputation_rate'] = (
        df_transformer['num_imputed_households'] / df_transformer['total_households']
    )

    return df_transformer


class TransformerDataPipeline:
    """Load and aggregate household data to transformer level"""

    def __init__(self, metadata, imp_folder, user_file_map):
        self.metadata = metadata
        self.imp_folder = Path(imp_folder)
        self.user_file_map = user_file_map
        self.transformer_groups = {}
        self.transformer_data = {}

    def create_transformer_groups(self):
        """Group users by zip_code (transformer)"""
        all_users = list(self.user_file_map.keys())
        zip_codes = sorted(self.metadata[Config.ZIP_CODE_COL].dropna().unique())

        # Distribute users across transformers
        # NOTE: Adjust this if you have actual user-to-transformer mapping
        for i, zip_code in enumerate(zip_codes):
            transformer_id = f"transformer_{int(zip_code)}"

            # Simple distribution: assign users evenly
            start_idx = i * len(all_users) // len(zip_codes)
            end_idx = (i + 1) * len(all_users) // len(zip_codes)
            self.transformer_groups[transformer_id] = all_users[start_idx:end_idx]

        print(f"\n✓ Created {len(self.transformer_groups)} transformer groups")
        for tid, users in list(self.transformer_groups.items())[:3]:
            print(f"  {tid}: {len(users)} users")

        return self.transformer_groups

    def load_user_data(self, user_id: str) -> pd.DataFrame:
        """Load individual user data"""
        if user_id not in self.user_file_map:
            return None

        filename = self.user_file_map[user_id]
        filepath = self.imp_folder / filename

        if not filepath.exists():
            return None

        try:
            df = pd.read_csv(filepath)
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp').reset_index(drop=True)

            # Rename columns
            df = df.rename(columns={
                'kWh': f'{user_id}_kWh',
                'imputed': f'{user_id}_imputed'
            })

            return df[['timestamp', f'{user_id}_kWh', f'{user_id}_imputed']]
        except Exception as e:
            print(f"  Warning: Error loading {filename}: {e}")
            return None

    def aggregate_transformer_load(self, transformer_id: str) -> pd.DataFrame:
        """Aggregate all users for a transformer"""
        if transformer_id not in self.transformer_groups:
            return None

        users = self.transformer_groups[transformer_id]

        dfs = []
        for user_id in users:
            df = self.load_user_data(user_id)
            if df is not None:
                dfs.append(df)

        if not dfs:
            return None

        # Merge all users
        df_merged = dfs[0]
        for df in dfs[1:]:
            df_merged = df_merged.merge(df, on='timestamp', how='outer')

        df_merged = df_merged.sort_values('timestamp').reset_index(drop=True)

        # Sum all kWh columns
        kwh_cols = [col for col in df_merged.columns if col.endswith('_kWh')]
        df_merged['transformer_load_kWh'] = df_merged[kwh_cols].sum(axis=1)

        # Calculate imputation stats
        df_merged = calculate_imputation_features(df_merged)
        df_merged['transformer_id'] = transformer_id

        return df_merged

    def process_all_transformers(self, sample_size: int = None):
        """Process all transformers"""
        if not self.transformer_groups:
            self.create_transformer_groups()

        transformer_ids = list(self.transformer_groups.keys())

        if sample_size:
            transformer_ids = transformer_ids[:sample_size]

        print(f"\nProcessing {len(transformer_ids)} transformers...")

        for i, tid in enumerate(transformer_ids):
            print(f"  {i+1}/{len(transformer_ids)}: {tid}", end='\r')
            df = self.aggregate_transformer_load(tid)
            if df is not None and len(df) > 0:
                self.transformer_data[tid] = df

        print(f"\n✓ Processed {len(self.transformer_data)} transformers")
        return self.transformer_data



class OverloadLabeler:
    """Label overload events"""

    def __init__(self, capacity_dict: Dict = None):
        self.capacity_dict = capacity_dict
        self.thresholds = {}

    def label_with_capacity(self, df: pd.DataFrame, transformer_id: str):
        """Use known capacity"""
        if self.capacity_dict and transformer_id in self.capacity_dict:
            capacity = self.capacity_dict[transformer_id]
            df['overload'] = (df['transformer_load_kWh'] > capacity).astype(int)
            df['capacity_utilization'] = df['transformer_load_kWh'] / capacity
        return df

    def label_with_statistical(self, df: pd.DataFrame, transformer_id: str,
                               percentile: float = 95, seasonal: bool = True):
        """Statistical threshold"""
        df = df.copy()

        if seasonal and len(df) > 1000:
            df['month'] = pd.to_datetime(df['timestamp']).dt.month
            df['season'] = df['month'].map({
                12: 'winter', 1: 'winter', 2: 'winter',
                3: 'spring', 4: 'spring', 5: 'spring',
                6: 'summer', 7: 'summer', 8: 'summer',
                9: 'fall', 10: 'fall', 11: 'fall'
            })

            thresholds = {}
            for season in df['season'].dropna().unique():
                season_data = df[df['season'] == season]['transformer_load_kWh']
                if len(season_data) > 0:
                    thresholds[season] = season_data.quantile(percentile / 100)

            if thresholds:
                df['threshold'] = df['season'].map(thresholds)
                df['overload'] = (df['transformer_load_kWh'] > df['threshold']).astype(int)
                self.thresholds[transformer_id] = thresholds
            else:
                seasonal = False

        if not seasonal:
            threshold = df['transformer_load_kWh'].quantile(percentile / 100)
            df['threshold'] = threshold
            df['overload'] = (df['transformer_load_kWh'] > threshold).astype(int)
            self.thresholds[transformer_id] = threshold

        return df

    def apply_to_all(self, data_dict: Dict, method: str = 'statistical', **kwargs):
        """Apply to all transformers"""
        labeled = {}
        for tid, df in data_dict.items():
            if method == 'capacity' and self.capacity_dict:
                labeled[tid] = self.label_with_capacity(df, tid)
            else:
                labeled[tid] = self.label_with_statistical(df, tid, **kwargs)
        return labeled



class FeatureEngineering:
    """Feature engineering"""

    @staticmethod
    def create_time_features(df):
        df = df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])

        df['hour'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.dayofweek
        df['month'] = df['timestamp'].dt.month
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['is_morning_peak'] = df['hour'].isin([7, 8, 9]).astype(int)
        df['is_evening_peak'] = df['hour'].isin([17, 18, 19, 20]).astype(int)

        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

        return df

    @staticmethod
    def create_lag_features(df, lags=[1, 2, 3, 24, 48, 168]):
        df = df.copy()
        for lag in lags:
            df[f'load_lag_{lag}h'] = df['transformer_load_kWh'].shift(lag)
        return df

    @staticmethod
    def create_rolling_features(df, windows=[3, 6, 12, 24]):
        df = df.copy()
        for window in windows:
            df[f'rolling_mean_{window}h'] = df['transformer_load_kWh'].rolling(window, min_periods=1).mean()
            df[f'rolling_std_{window}h'] = df['transformer_load_kWh'].rolling(window, min_periods=1).std()
            df[f'rolling_max_{window}h'] = df['transformer_load_kWh'].rolling(window, min_periods=1).max()
        return df

    @staticmethod
    def create_change_features(df):
        df = df.copy()
        df['load_diff_1h'] = df['transformer_load_kWh'].diff(1)
        df['load_pct_change_1h'] = df['transformer_load_kWh'].pct_change(1)
        return df

    @staticmethod
    def create_all_features(df):
        df = FeatureEngineering.create_time_features(df)
        df = FeatureEngineering.create_lag_features(df, Config.LAG_HOURS)
        df = FeatureEngineering.create_rolling_features(df, Config.ROLLING_WINDOWS)
        df = FeatureEngineering.create_change_features(df)
        return df


class DataSplitter:
    """Time series split"""

    @staticmethod
    def split_data(df, train_r=0.7, val_r=0.15, horizon=1):
        df = df.sort_values('timestamp').reset_index(drop=True)

        n = len(df)
        train_end = int(n * train_r)
        val_end = int(n * (train_r + val_r))

        df['target'] = df['overload'].shift(-horizon)
        df = df[:-horizon]

        return df.iloc[:train_end], df.iloc[train_end:val_end], df.iloc[val_end:]

    @staticmethod
    def prepare_xy(df, feature_cols):
        df_clean = df.dropna(subset=feature_cols + ['target'])
        X = df_clean[feature_cols]
        y = df_clean['target']

        print(f"  Features: {len(feature_cols)}, Samples: {len(X)}, Overload: {y.mean():.2%}")
        return X, y, df_clean['timestamp']


from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

class ModelTrainer:
    """XGBoost model"""

    def __init__(self):
        self.model = None
        self.feature_importance = None

    def build_model(self):
        self.model = XGBClassifier(
            n_estimators=200,
            max_depth=7,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
        return self.model

    def train(self, X_train, y_train, X_val=None, y_val=None):
        if self.model is None:
            self.build_model()

        print("\nTraining XGBoost...")

        if X_val is not None:
            self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        else:
            self.model.fit(X_train, y_train)

        self.feature_importance = pd.DataFrame({
            'feature': X_train.columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)

        print("✓ Training complete!")

    def evaluate(self, X, y, name='Test'):
        print(f"\n{'='*60}")
        print(f"Evaluation: {name}")
        print(f"{'='*60}")

        y_pred = self.model.predict(X)
        y_proba = self.model.predict_proba(X)[:, 1]

        print("\n" + classification_report(y, y_pred, target_names=['Normal', 'Overload']))

        cm = confusion_matrix(y, y_pred)
        print(f"Confusion Matrix:\n{cm}")

        try:
            auc = roc_auc_score(y, y_proba)
            print(f"\nROC-AUC: {auc:.4f}")
        except:
            auc = None

        return {'predictions': y_pred, 'probabilities': y_proba,
                'roc_auc': auc, 'confusion_matrix': cm}

print("✓ All classes loaded!")

✓ All classes loaded!


In [9]:
print("\n" + "="*80)
print("STEP 3: PROCESSING TRANSFORMER DATA")
print("="*80)

pipeline = TransformerDataPipeline(meta_data, Config.IMP_FOLDER, user_file_map)
transformer_data = pipeline.process_all_transformers(Config.SAMPLE_TRANSFORMERS)

if transformer_data:
    sample_tid = list(transformer_data.keys())[0]
    sample_df = transformer_data[sample_tid]

    print(f"\nSample: {sample_tid}")
    print(f"  Rows: {len(sample_df)}")
    print(f"  Date: {sample_df['timestamp'].min()} to {sample_df['timestamp'].max()}")
    print(f"  Mean load: {sample_df['transformer_load_kWh'].mean():.2f} kWh")
    print(f"  Max load: {sample_df['transformer_load_kWh'].max():.2f} kWh")
else:
    print("❌ No data loaded!")


STEP 3: PROCESSING TRANSFORMER DATA

✓ Created 434 transformer groups
  transformer_1001: 8 users
  transformer_1002: 9 users
  transformer_1003: 9 users

Processing 434 transformers...

✓ Processed 434 transformers

Sample: transformer_1001
  Rows: 24144
  Date: 2017-05-30 01:00:00 to 2020-03-01 00:00:00
  Mean load: 2.36 kWh
  Max load: 10.53 kWh


In [10]:
print("\n" + "="*80)
print("STEP 3: PROCESSING TRANSFORMER DATA")
print("="*80)

pipeline = TransformerDataPipeline(meta_data, Config.IMP_FOLDER, user_file_map)
transformer_data = pipeline.process_all_transformers(Config.SAMPLE_TRANSFORMERS)

if transformer_data:
    sample_tid = list(transformer_data.keys())[0]
    sample_df = transformer_data[sample_tid]

    print(f"\nSample: {sample_tid}")
    print(f"  Rows: {len(sample_df)}")
    print(f"  Date: {sample_df['timestamp'].min()} to {sample_df['timestamp'].max()}")
    print(f"  Mean load: {sample_df['transformer_load_kWh'].mean():.2f} kWh")
    print(f"  Max load: {sample_df['transformer_load_kWh'].max():.2f} kWh")
else:
    print("❌ No data loaded!")




STEP 3: PROCESSING TRANSFORMER DATA

✓ Created 434 transformer groups
  transformer_1001: 8 users
  transformer_1002: 9 users
  transformer_1003: 9 users

Processing 434 transformers...

✓ Processed 434 transformers

Sample: transformer_1001
  Rows: 24144
  Date: 2017-05-30 01:00:00 to 2020-03-01 00:00:00
  Mean load: 2.36 kWh
  Max load: 10.53 kWh


In [ ]:
print("\n" + "="*80)
print("STEP 4: LABELING & FEATURE ENGINEERING")
print("="*80)

if not transformer_data:
    print("❌ No data to process")
else:
    # Create capacity dictionary
    capacities = {}
    for _, row in zip_group.iterrows():
        tid = f"transformer_{int(row[Config.ZIP_CODE_COL])}"
        capacities[tid] = row['transformer_capacity_kW']

    # Label overloads
    print("\nLabeling overloads...")
    labeler = OverloadLabeler(capacity_dict=capacities)
    labeled_data = labeler.apply_to_all(
        transformer_data,
        method=Config.OVERLOAD_METHOD,
        percentile=Config.STATISTICAL_PERCENTILE,
        seasonal=Config.USE_SEASONAL_THRESHOLDS
    )

    print(f"✓ Labeled {len(labeled_data)} transformers")

    # Engineer features
    print("\nEngineering features...")
    all_data = []
    for tid, df in labeled_data.items():
        df_feat = FeatureEngineering.create_all_features(df)
        all_data.append(df_feat)

    df_combined = pd.concat(all_data, ignore_index=True)
    print(f"✓ Combined: {len(df_combined)} samples")

    # Check overload rate
    overload_rate = df_combined['overload'].mean()
    print(f"✓ Overall overload rate: {overload_rate:.2%}")


STEP 4: LABELING & FEATURE ENGINEERING

Labeling overloads...
✓ Labeled 434 transformers

Engineering features...


In [ ]:
print("\n" + "="*80)
print("STEP 5: TRAINING MODEL")
print("="*80)

if 'df_combined' not in locals():
    print("❌ No data available")
else:
    # Split data
    print("\nSplitting data...")
    splitter = DataSplitter()
    train_df, val_df, test_df = splitter.split_data(
        df_combined,
        train_r=Config.TRAIN_RATIO,
        val_r=Config.VAL_RATIO,
        horizon=Config.PREDICTION_HORIZON
    )

    print(f"✓ Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    # Define features
    exclude = ['timestamp', 'transformer_id', 'overload', 'target',
               'threshold', 'season', 'month']
    exclude += [c for c in df_combined.columns if c.endswith('_kWh') and 'transformer_load' not in c]
    exclude += [c for c in df_combined.columns if c.endswith('_imputed')]

    feature_cols = [c for c in train_df.columns if c not in exclude]

    print(f"\n✓ Features: {len(feature_cols)}")

    # Prepare datasets
    print("\nPreparing datasets...")
    X_train, y_train, _ = splitter.prepare_xy(train_df, feature_cols)
    X_val, y_val, _ = splitter.prepare_xy(val_df, feature_cols)
    X_test, y_test, ts_test = splitter.prepare_xy(test_df, feature_cols)

    # Train
    print("\n" + "="*60)
    model = ModelTrainer()
    model.train(X_train, y_train, X_val, y_val)

    # Evaluate
    val_results = model.evaluate(X_val, y_val, 'Validation')
    test_results = model.evaluate(X_test, y_test, 'Test')

In [ ]:
print("\n" + "="*80)
print("STEP 6: SAVING RESULTS")
print("="*80)

if 'model' in locals():
    # Save model
    model_path = f'{Config.OUTPUT_DIR}/model_xgboost_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model.model, f)
    print(f"✓ Model: {model_path}")

    # Save feature importance
    importance_path = f'{Config.OUTPUT_DIR}/feature_importance.csv'
    model.feature_importance.to_csv(importance_path, index=False)
    print(f"✓ Importance: {importance_path}")

    print("\nTop 10 Features:")
    print(model.feature_importance.head(10).to_string(index=False))

    # Save predictions
    predictions_df = pd.DataFrame({
        'timestamp': ts_test.values,
        'actual': y_test.values,
        'predicted': test_results['predictions'],
        'probability': test_results['probabilities']
    })
    pred_path = f'{Config.OUTPUT_DIR}/predictions.csv'
    predictions_df.to_csv(pred_path, index=False)
    print(f"✓ Predictions: {pred_path}")

    print("\n" + "="*80)
    print("✅ PIPELINE COMPLETE!")
    print("="*80)
    print(f"\nFinal ROC-AUC: {test_results['roc_auc']:.4f}")

In [ ]:
print("\n" + "="*80)
print("STEP 7: VISUALIZATIONS")
print("="*80)

if 'model' in locals() and model.feature_importance is not None:

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Feature Importance
    ax = axes[0, 0]
    top = model.feature_importance.head(15)
    ax.barh(range(len(top)), top['importance'])
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['feature'])
    ax.set_xlabel('Importance')
    ax.set_title('Top 15 Features')
    ax.invert_yaxis()

    # Probability Distribution
    ax = axes[0, 1]
    normal = predictions_df[predictions_df['actual']==0]['probability']
    overload = predictions_df[predictions_df['actual']==1]['probability']
    ax.hist(normal, bins=50, alpha=0.7, label='Normal')
    ax.hist(overload, bins=50, alpha=0.7, label='Overload')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Count')
    ax.set_title('Prediction Distribution')
    ax.legend()

    # Confusion Matrix
    ax = axes[1, 0]
    cm = test_results['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title('Confusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

    # Time Series
    ax = axes[1, 1]
    sample = predictions_df.head(500)
    ax.plot(sample['timestamp'], sample['probability'], alpha=0.7, label='Probability')
    overload_points = sample[sample['actual']==1]
    ax.scatter(overload_points['timestamp'], overload_points['probability'],
              color='red', label='Actual Overload', s=20)
    ax.axhline(y=0.5, color='r', linestyle='--', label='Threshold')
    ax.set_xlabel('Time')
    ax.set_ylabel('Probability')
    ax.set_title('Predictions Over Time (Sample)')
    ax.legend()
    plt.xticks(rotation=45)

    plt.tight_layout()
    viz_path = f'{Config.OUTPUT_DIR}/visualization.png'
    plt.savefig(viz_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✓ Saved: {viz_path}")